# Classification Metrics

In 09b we introduced the vocabulary of classification evaluation: precision, recall, F1, and the confusion matrix. We used those metrics to compare classifiers on MNIST and saw that accuracy alone can be misleading.

This notebook goes deeper. We'll visualize the *entire* precision-recall tradeoff with curves, introduce the ROC curve as a threshold-agnostic evaluation tool, and set up the cost-sensitive framing that drives Thursday's lecture.

We continue with the MNIST binary "5 vs not-5" task from 09b.

## Setup

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    PrecisionRecallDisplay,
    average_precision_score,
    roc_curve,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load MNIST - same setup as 09b
mnist = fetch_openml('mnist_784', as_frame=False)
X, y = mnist.data, mnist.target

# Predefined train/test split
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

# Binary labels: 5 vs not-5
y_train_5 = (y_train == '5')
y_test_5 = (y_test == '5')

print(f"Training: {X_train.shape[0]:,} samples ({y_train_5.mean():.1%} are 5s)")
print(f"Test:     {X_test.shape[0]:,} samples ({y_test_5.mean():.1%} are 5s)")

## Baseline Model

We fit the same Pipeline from 09b: `StandardScaler` → `LogisticRegression`. This gives us a well-performing model to evaluate with the new tools.

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train_5)

print(f"Training accuracy: {logreg.score(X_train, y_train_5):.4f}")
print(f"Test accuracy:     {logreg.score(X_test, y_test_5):.4f}")

In [ ]:
y_pred = logreg.predict(X_test)
print(classification_report(y_test_5, y_pred))

High accuracy, but recall for the "True" class (actual 5s) is noticeably lower than precision. The model misses some 5s. In 09b we saw this pattern with the DummyClassifier - now we have the tools to visualize *why* and decide what to do about it.

##### How would you explain, in plain English, the first statement in the previous paragraph?

The model is very good at identifying non-5s (the majority class). When it predicts something is a 5, it's usually right (high precision). But it doesn't catch all the actual 5s - some slip through and get labeled as not-5 (lower recall).

In concrete terms: if there are 892 actual 5s in the test set and the model has 84% recall, it correctly identifies about 749 of them (892 * 0.84) but misses ~143 (892 - 749). Those 143 are false negatives - observations where the model predicted 'not-5' where truth was '5'.

## Two Kinds of Classifier Scores

It would be helpful to visualize precision, recall, and decision scores / probabilities in various ways. In 09b we looked at how precision vs recall varied across three thresholds. What does it look like at *every* point?

Before we build those plots, we need to understand the raw material they're made of. When a classifier makes a prediction, it doesn't jump straight to a class label - it first computes a continuous score, then applies a threshold to decide the label. sklearn exposes two methods for accessing these scores.

In [ ]:
# Same sample, same model, two different score types
sample = X_test[:1]

clf = logreg.named_steps['logisticregression']
scaler = logreg.named_steps['standardscaler']
sample_s = scaler.transform(sample)

# grab one score and one probability (pair, sums to 1) from the fitted logreg model
df_score = clf.decision_function(sample_s)[0]
proba = clf.predict_proba(sample_s)[0]

print(f"decision_function: {df_score:.4f}")
print(f"predict_proba:     {proba}")
print(f"  P(not 5) = {proba[0]:.4f}")
print(f"  P(is 5)  = {proba[1]:.4f}")

| | `decision_function` | `predict_proba` |
|---|---|---|
| Scale | Unbounded ($-\infty$ to $+\infty$) | [0, 1] |
| Default threshold | 0 | 0.5 |
| Interpretation | Signed distance from decision boundary | Estimated class probability |
| Available for | LogReg, SVM, linear models | Most classifiers (KNN, LogReg, trees) |
| Not available for | KNN | - |

For LogisticRegression, the two are related by the logistic function: `predict_proba` = $\sigma$(`decision_function`). A `decision_function` value of 0 maps to `predict_proba` = 0.5 - they represent the same decision boundary, just on different scales.

Recall from 08b that the logistic function maps the linear combination $z = w \cdot x + b$ to a probability. `decision_function` returns that $z$ value; `predict_proba` applies the logistic function to convert it to $P(\text{positive})$. Same information, different scale.

In 09b we used `predict_proba` to explore thresholds. The curves below work with *either* score type - all they need is a way to rank samples from "most likely positive" to "least likely positive." We'll use `decision_function` for LogReg (the scores are more spread out, which makes the plots easier to read) and `predict_proba` for KNN (which doesn't have a `decision_function`).

## Getting Honest Scores with `cross_val_predict`

To get a score (or prob) for every training sample. We could call `logreg.decision_function(X_train)` directly, but that model was trained on `X_train` - it already saw those samples and `decision_function` will calculate the score based on the fitted parameters. The scores would be optimistically biased, just like reporting training accuracy instead of test accuracy.

`cross_val_predict` gives us honest scores using the same fold-based strategy as `cross_val_score`:

1. Split `X_train` into 5 folds
2. Train on folds 2-5, compute `decision_function` scores for fold 1
3. Train on folds 1,3-5, compute scores for fold 2
4. ... repeat for all 5 folds
5. Concatenate: every sample now has a score from a model that *never saw it during training*

In [ ]:
# Get decision_function scores via 5-fold CV
# Each sample gets a score from a model that was NOT trained on it
y_scores = cross_val_predict(
    logreg, X_train, y_train_5, cv=5, method="decision_function"
)

print(f"Shape: {y_scores.shape}")
print(f"Score range: [{y_scores.min():.2f}, {y_scores.max():.2f}]")
print(f"Score for first sample (label={y_train_5[0]}): {y_scores[0]:.4f}")

Despite the function name, `cross_val_predict` returns whatever `method` you ask for - class predictions (default), `decision_function` scores, or `predict_proba` probabilities. Here we get 60,000 honest decision scores, one per training sample.

> **Important caveat:** each score comes from a *different* model (trained on a different 80% of the data). These are five separate LogisticRegression fits, not one. The scores are comparable for building curves, but they aren't predictions from a single trained model.

## Precision-Recall Curve

Following up on the original example, `precision_recall_curve` allows us to see the tradeoff across all possible thresholds.

It takes the true labels and the continuous scores, then sweeps through all possible threshold values. At each threshold, it computes: "if we classify everything above this score as positive, what are the resulting precision and recall?"

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)

print(f"Number of thresholds evaluated: {len(thresholds):,}")

In [ ]:
# Precision and recall as functions of the decision score threshold
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, precisions[:-1], "b--", label="Precision", linewidth=2)
ax.plot(thresholds, recalls[:-1], "g-", label="Recall", linewidth=2)
ax.set_xlabel("Threshold (decision_function score)")
ax.set_ylabel("Score")
ax.set_title("Precision and Recall vs. Threshold")
ax.legend(loc="center right")
ax.set_xlim([-20, 20])
ax.set_ylim([0, 1.05])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Raising the threshold increases precision (fewer false alarms) but decreases recall (more missed 5s). Lowering it does the opposite. There is no free lunch - you're always trading one for the other.

The crossover point is where precision = recall, which corresponds to the "balanced" operating point. This falls near a score of 0, which is not a coincidence - for LogisticRegression, a `decision_function` score of 0 *is* the decision boundary (equivalent to `predict_proba` = 0.5). The default threshold already roughly balances precision and recall.

### The Precision-Recall Curve

An equivalent and often more useful view plots precision directly against recall. Each point on this curve corresponds to one threshold, but the threshold axis is implicit - we just see the tradeoff.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
PrecisionRecallDisplay.from_predictions(
    y_train_5, y_scores, name="LogReg", ax=ax
)
ax.set_title("Precision-Recall Curve")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

A perfect classifier would hug the top-right corner (precision=1, recall=1). The further the curve drops, the harder the tradeoff.

### Average Precision (AP)

How do we summarize a curve with a single number? With the area under it.

`average_precision_score` computes the area under the precision-recall curve (hence the name - it's a weighted average of precision at each recall level). Think of it as a single number that captures the overall quality of the precision-recall tradeoff - higher is better.

In [ ]:
ap = average_precision_score(y_train_5, y_scores)
print(f"Average Precision: {ap:.4f}")

### Finding a Threshold for a Target Metric

Suppose we need at least 95% precision. We can find the corresponding threshold using the arrays returned by `precision_recall_curve`. Those arrays are sorted by increasing threshold, so precision generally increases (and recall decreases) as we move through the array. The `argmax` trick finds the first index where our condition is met:

In [ ]:
# Find the lowest threshold that achieves at least 95% precision
target_precision = 0.95

# argmax() on a boolean array returns the index of the first True value.
# (precisions >= 0.95) is [False, False, ..., True, True, ...] and
# argmax() finds where it first becomes True.
idx_95 = (precisions >= target_precision).argmax()
thresh_95 = thresholds[idx_95]

print(f"Threshold for {target_precision:.0%} precision: {thresh_95:.4f}")
print(f"Precision at that threshold: {precisions[idx_95]:.4f}")
print(f"Recall at that threshold:    {recalls[idx_95]:.4f}")

High precision came at a cost - recall dropped significantly. To make predictions at this threshold:

In [ ]:
# Apply the custom threshold to the out-of-fold scores
y_pred_95 = (y_scores >= thresh_95)

print(f"Default predictions (threshold=0):")
print(classification_report(y_train_5, (y_scores >= 0), digits=3))

print(f"High-precision predictions (threshold={thresh_95:.2f}):")
print(classification_report(y_train_5, y_pred_95, digits=3))

This is the manual version of what `FixedThresholdClassifier` does in a Pipeline (09b). Both achieve the same thing - the choice depends on whether you need the threshold to be part of the sklearn estimator API.

## ROC Curve

The precision-recall curve shows the tradeoff from the *positive class* perspective. The ROC (Receiver Operating Characteristic) curve shows it from a different angle: True Positive Rate (recall) vs. False Positive Rate.

$$\text{TPR} = \frac{TP}{TP + FN} = \text{Recall}$$

$$\text{FPR} = \frac{FP}{FP + TN}$$

- TPR (recall): "of all actual positives, what fraction did we catch?"
- FPR: "of all actual negatives, what fraction did we incorrectly flag as positive?"

Think of FPR as the false alarm rate. TPR / FPR are complementary - same question but about actual positives vs negatives.

As we lower the threshold (classify more samples as positive), *both* TPR and FPR increase: we catch more real positives, but we also trigger more false alarms on negatives. The ROC curve traces this tradeoff using the same continuous scores we computed above.

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_train_5, y_scores)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(
    y_train_5, y_scores, name="LogReg", ax=ax
)
ax.plot([0, 1], [0, 1], "k:", label="Random classifier")
ax.set_title("ROC Curve")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The diagonal represents a random classifier - it gains true positives and false positives at the same rate. A good model hugs the top-left corner: high TPR (catching positives) with low FPR (few false alarms).

### ROC AUC

The Area Under the ROC Curve (AUC) summarizes the entire curve as a single number.

In [ ]:
auc = roc_auc_score(y_train_5, y_scores)
print(f"ROC AUC: {auc:.4f}")

AUC ranges from 0 to 1:
- 1.0 = perfect separation
- 0.5 = random (the diagonal)
- Below 0.5 = worse than random, which usually means the labels are inverted - a useful diagnostic, not a disaster

AUC has a concrete probabilistic interpretation: pick one random positive sample and one random negative sample. AUC is the probability that the model assigns a higher score to the positive. An AUC of 0.97 means that if you randomly select one actual 5 and one actual non-5, there is a 97% chance the model gives a higher score to the 5.

This makes AUC a measure of *ranking quality* - how well the model separates positives from negatives - independent of any particular threshold. This is useful for comparing models, but it **can be misleading for imbalanced data**.

Because FPR is computed relative to the large negative class, even a small FPR can correspond to many false positives in absolute terms. A model can achieve high AUC while still performing poorly on the minority class.

### When to Use Which Curve

| Situation | Prefer | Why |
|-----------|--------|-----|
| Balanced classes | ROC / AUC | FPR is meaningful when negatives are common |
| Imbalanced classes | PR / AP | Focuses on the positive (minority) class |
| Comparing models overall | ROC AUC | Threshold-independent, widely understood |
| Optimizing for a business cost | PR curve | Directly shows precision-recall tradeoff |

In practice, report both. They tell complementary stories.

## Model Comparison: LogReg vs KNN

Different classifiers draw different decision boundaries. LogisticRegression draws a linear boundary (high bias, low variance). KNN adapts to local structure (low bias, high variance). Let's compare them using the curves we just learned.

In [ ]:
# Fit KNN pipeline
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn.fit(X_train, y_train_5)

print(f"KNN test accuracy: {knn.score(X_test, y_test_5):.4f}")

KNN doesn't have a `decision_function` - it estimates probabilities directly from neighbor votes. So we use `cross_val_predict` with `method="predict_proba"` instead. The curve-building functions don't care which score type they receive; they just need a way to rank samples.

In [ ]:
# Out-of-fold probability estimates for KNN
y_proba_knn = cross_val_predict(
    knn, X_train, y_train_5, cv=5, method="predict_proba"
)[:, 1]  # P(is a 5)

print(f"Score range: [{y_proba_knn.min():.3f}, {y_proba_knn.max():.3f}]")

### ROC Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(
    y_train_5, y_scores, name="LogReg", ax=ax
)
RocCurveDisplay.from_predictions(
    y_train_5, y_proba_knn, name="KNN (k=5)", ax=ax
)
ax.plot([0, 1], [0, 1], "k:", label="Random")
ax.set_title("ROC Curve Comparison")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
auc_logreg = roc_auc_score(y_train_5, y_scores)
auc_knn = roc_auc_score(y_train_5, y_proba_knn)

print(f"LogReg ROC AUC: {auc_logreg:.4f}")
print(f"KNN    ROC AUC: {auc_knn:.4f}")

KNN dominates on the ROC curve, especially in the low-FPR region (left side of the plot). Its flexible, locally-adaptive boundary captures digit patterns that a linear model misses. The AUC gap is modest (0.97 vs 0.99) but consistent across the entire curve.

### PR Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
PrecisionRecallDisplay.from_predictions(
    y_train_5, y_scores, name="LogReg", ax=ax
)
PrecisionRecallDisplay.from_predictions(
    y_train_5, y_proba_knn, name="KNN (k=5)", ax=ax
)
ax.set_title("Precision-Recall Curve Comparison")
ax.legend(loc="lower left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
ap_logreg = average_precision_score(y_train_5, y_scores)
ap_knn = average_precision_score(y_train_5, y_proba_knn)

print(f"LogReg Average Precision: {ap_logreg:.4f}")
print(f"KNN    Average Precision: {ap_knn:.4f}")

The PR curve tells a more dramatic story than the ROC curve. KNN maintains near-perfect precision across almost the entire recall range, only dropping off sharply past ~95% recall. LogReg's precision starts declining much sooner. The AP gap (0.89 vs 0.95) is larger than the AUC gap - PR curves are more sensitive to differences in minority-class performance, which is why they're preferred for imbalanced problems.

Notice that the two curves agree on which model is better, but they disagree on *how much* better. This is why reporting a single number (accuracy, F1, or even AUC) is never the whole story.

## The Cost Matrix: Bridge to Thursday

Every classification problem has an implicit cost matrix, even if nobody writes it down.

|  | Predicted Negative | Predicted Positive |
|---|---|---|
| Actually Negative | TN: correct | FP: cost of false alarm |
| Actually Positive | FN: cost of missed positive | TP: correct (or benefit) |

The default 0.5 threshold assumes FP and FN cost the same. When they don't - and they almost never do - the "right" threshold is a function of the cost ratio, not a convention.

Consider two scenarios:

*Spam filter:* a false positive (good email sent to spam) can be a real problem - you might miss an important message. A false negative (spam in inbox) is a minor nuisance. Costs are asymmetric → the default threshold may be too aggressive.

*Medical screening:* a false positive (unnecessary follow-up test) costs time and money. A false negative (missed disease) costs a life. Costs are wildly asymmetric → threshold should be much lower than 0.5.

The PR and ROC curves show us *all possible* operating points. The cost matrix tells us which point is *right*. On Thursday we'll make those costs explicit in dollars and let sklearn find the optimal threshold automatically.

## Summary

- Classifiers produce continuous scores before applying a threshold to predict a class label. `decision_function` gives unbounded scores (threshold at 0); `predict_proba` gives probabilities (threshold at 0.5). The curve-building tools accept either.
- `cross_val_predict` gives honest, out-of-fold scores for every training sample - but remember each score comes from a different model.
- `precision_recall_curve` and `PrecisionRecallDisplay` show the full precision-recall tradeoff across all thresholds.
- `average_precision_score` summarizes the PR curve as a single number - prefer it over ROC AUC for imbalanced data.
- `roc_curve`, `roc_auc_score`, and `RocCurveDisplay` provide threshold-agnostic model comparison. AUC is the probability that a random positive gets a higher score than a random negative.
- Every classification problem has an implicit cost matrix. Making it explicit turns threshold selection from art into arithmetic.

Next: we put dollar amounts on misclassification costs and let sklearn find the optimal threshold.